# Advanced Exploratory Data Analysis (EDA)
## AI-Driven Polyglot Middleware Training Dataset

**MSc Thesis in Computer Science**  
University of Port Harcourt  
Author: [Your Full Name]  
Date: June 2026

**Objective**: Analyze the training data, feature distributions, model performance, and explainability using SHAP values.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap
from pathlib import Path
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

sns.set(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (14, 8)
%matplotlib inline

## 1. Loading the Dataset

In [ ]:
# Load latest versioned dataset
data_files = sorted(Path("app/ai").glob("real_training_data_*.csv"))
df = pd.read_csv(data_files[-1])

print(f"✅ Dataset Loaded: {data_files[-1].name}")
print(f"Shape: {df.shape}")
print("\nTarget Distribution:")
print(df['target'].value_counts())

## 2. Feature Statistics & Distributions

In [ ]:
features = ['scs', 'max_depth', 'transactional_score', 'cache_score',
            'relationship_score', 'time_series_score', 'schema_flexibility']

print("Feature Statistics:")
print(df[features].describe())

df[features].hist(bins=25, figsize=(16, 10))
plt.suptitle('Feature Distributions', fontsize=16)
plt.show()

## 3. Correlation Analysis

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df[features].corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.show()

## 4. Feature Distribution by Database Type

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
for i, feature in enumerate(features):
    sns.boxplot(x='target', y=feature, data=df, ax=axes[i//4, i%4])
    axes[i//4, i%4].set_xticklabels(axes[i//4, i%4].get_xticklabels(), rotation=45)
plt.suptitle('Feature Distribution by Target Database')
plt.tight_layout()
plt.show()

## 5. Model Performance Comparison

In [ ]:
try:
    metrics = joblib.load("app/ai/models/metrics.pkl")
    metrics_df = pd.DataFrame(metrics).T
    print("Model Performance:")
    print(metrics_df)

    metrics_df['accuracy'].sort_values(ascending=False).plot(kind='bar', color='skyblue')
    plt.title('Model Accuracy Comparison')
    plt.ylabel('Accuracy')
    plt.xticks(rotation=45)
    plt.show()
except:
    print("Train models first to view metrics.")

## 6. SHAP Explainability (Feature Importance)

In [ ]:
# Load the best model (Random Forest is usually good for SHAP)
try:
    model = joblib.load("app/ai/models/random_forest.pkl")
    X = df.drop("target", axis=1)
    
    # SHAP Explainer
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X.sample(100))  # Sample for speed

    # Summary Plot
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X.sample(100), plot_type="bar")
    plt.title("SHAP Feature Importance (Random Forest)")
    plt.show()

    print("✅ SHAP analysis completed.")
except Exception as e:
    print(f"SHAP analysis failed: {e}. Make sure models are trained.")